In [280]:
import sys
sys.path.append('./Textual-Anomaly-Detection-Framework/Anomaly Detection Framework')

from Data_Preparation.Embedding import embedding_encoder
from Data_Preparation.Tac import tac
from Data_Preparation import utils
from Modelisation.FlowMatching import flow_matching
from Modelisation.Baselines.OCSVM import ocsvm
from Modelisation.Baselines.CVDD.utils import build_vocab, cvdd_model_pipeline
import Modelisation.evaluation as ev
from Modelisation.Baselines.CVDD.networks import cvdd_Net
from utils import save_results


import torch
from torch import Tensor
from torch.utils.data import TensorDataset, DataLoader
import optuna
import torch
from torch import nn, Tensor
import numpy as np
from transformers import AutoTokenizer
from datasets import Dataset, concatenate_datasets
import time

BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [213]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [259]:
# train_20ng_, test_20ng_ = utils.import_dataset(name="20newsgroups", batch_size=BATCH_SIZE)
# train_reuters_, test_reuters_ = utils.import_dataset(name="reuters", batch_size=BATCH_SIZE)
# train_wos = utils.import_dataset(name="WOS", batch_size=BATCH_SIZE)
# train_dbpedia14, test_dbpedia14 = utils.import_dataset(name="DBpedia14", batch_size=BATCH_SIZE)
train_agnews_, test_agnews_ = utils.import_dataset(name="agnews", batch_size=BATCH_SIZE)

agnews dataset importing .... 




In [260]:
train_agnews_ = utils.preprocess(train_agnews_.dataset)
test_agnews_ = utils.preprocess(test_agnews_.dataset)

In [271]:
def train_test_val_split(train, test, inlier_topic, dataset_name, type_tac, anomaly_rate, verbose=False):
    
    train_inlier, train_anomaly = tac.textual_anomaly_contamination(train, dataset_name, inlier_topic, type_tac, anomaly_rate, True)

    n_inliers_val = int(0.1 * len(train_inlier))
    inlier_indices = np.random.choice(len(train_inlier), n_inliers_val, replace=False)
    val_inlier_dataset = train_inlier.select(inlier_indices)

    train_inlier = train_inlier.select([i for i in range(len(train_inlier)) if i not in inlier_indices])

    n_anomalies_val = int(n_inliers_val / 0.9 * 0.1)
    anomaly_indices = np.random.choice(len(train_anomaly), n_anomalies_val, replace=False)
    val_anomaly_dataset = train_anomaly.select(anomaly_indices)

    val_ = concatenate_datasets([val_inlier_dataset, val_anomaly_dataset]).shuffle(seed=42)
    
    if verbose:
        print("TRAINSET")
        print(train_inlier)
        print(train_anomaly)
    
    if verbose:
        print("\nVALSET")
        print(val_)
        print()

    test_ = tac.textual_anomaly_contamination(test, dataset_name, inlier_topic, type_tac, anomaly_rate, False)
    if verbose:
        print("TESTSET")
        print(test_)

    return train_inlier, train_anomaly, val_, test_

In [272]:
inlier_topic = 'World'
dataset_name = 'agnews'
type_tac = 'fate' 
anomaly_rate = 0.1

train_inlier_agnews, train_anomaly_agnews, val_agnews, test_agnews = train_test_val_split(train_agnews_, test_agnews_, inlier_topic, dataset_name, type_tac, anomaly_rate, True)

TRAINSET
Dataset({
    features: ['text', 'label', 'anomaly_class'],
    num_rows: 27000
})
Dataset({
    features: ['text', 'label', 'anomaly_class'],
    num_rows: 3333
})

VALSET
Dataset({
    features: ['text', 'label', 'anomaly_class'],
    num_rows: 3333
})

TESTSET
Dataset({
    features: ['text', 'label', 'anomaly_class'],
    num_rows: 2111
})


In [273]:
model_name = 'all-MiniLM-L6-v2'
sentencebertEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'sentencebert', device)

# model_name = 'distilbert-base-uncased'

# bertEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'bert')

# model_name = 'glove_300d.kv'

# gloveEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'glove')

In [276]:
train_inlier_agnews = sentencebertEncoder.forward(train_inlier_agnews)
# train_anomaly_dl_20ng = sentencebertEncoder.forward(train_anomaly_dl_20ng)

test_agnews = sentencebertEncoder.forward(val_agnews)
# test_dl_20ng = sentencebertEncoder.forward(test_dl_20ng)

val_agnews = sentencebertEncoder.forward(test_agnews)
# val_20ng = sentencebertEncoder.forward(val_20ng

In [277]:
# X_inlier = Tensor(train_inlier_dl_20ng['sbert_embeddings']).to(device)
X_inlier = Tensor(train_inlier_agnews['sbert_embeddings']).to(device)
# X_inlier = Tensor(train_inlier_dl_20ng['glove_embedding']).to(device)

X_test =  Tensor(test_agnews['sbert_embeddings']).to(device)
y_test = np.array(test_agnews['anomaly_class'])

X_val =  Tensor(val_agnews['sbert_embeddings']).to(device)
y_val = np.array(val_agnews['anomaly_class'])

print(X_inlier.shape)
print(X_test.shape)
print(y_test.shape)
print(X_val.shape)
print(y_val.shape)

torch.Size([27000, 384])
torch.Size([3333, 384])
(3333,)
torch.Size([3333, 384])
(3333,)


## FM 

In [233]:
batch_size_default = 32
X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size_default, shuffle=True)
input_dim = X_inlier.shape[1]
latent_dim = 256
sinu = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def objective(trial):

    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    # n_epochs = trial.suggest_int("n_epochs", 1000, 2000, step=100)
    n_epochs = trial.suggest_int("n_epochs", 100, 500, step=50)
    # source = trial.suggest_categorical("source", ["sphere", "sphere-noised"])
    source = trial.suggest_categorical("source", ["gaussian", "sphere", "sphere-noised"])
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0, 1e-3, log=False)

    dl_train = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

    flow_model = flow_matching.FlowMatching(source, X_inlier.cpu(), input_dim, latent_dim, sinu, device).to(device)

    optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    print("\n############################################################")
    print(f"batch_size: {batch_size} | n_epochs:{n_epochs} | source: {source} | lr: {lr} | weight_decay: {weight_decay}\n")
    fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=True)
    flow_model_trained = fm_trainer.train(dl_train, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')
    

    auc, fpr95, ap = fm_trainer.test(X_val, y_val, score_type='norm', solver_type='midpoint', n_steps=10)

    score = auc + ap - fpr95
    
    print(f"FM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f} | SCORE: {score: .4f}")  
    
#     ocsvm_kwargs = {
#         "nu": 0.1,
#         "kernel": 'rbf',
#         "gamma": 'scale'
#         }
#     clf, _, _ = ocsvm.One_Class_SVM(X_inlier.cpu().detach(), ocsvm_kwargs)

#     _ = clf.predict(X_test.cpu().detach())           
#     scores_val = clf.decision_function(X_val.cpu().detach())

#     auc, ap, fpr95 = ev.evaluation(y_val, scores_val, verbose=False)
#     print(f"OCSVM --> AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")
    
    return score


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)  

print("Best hyperparameters:", study.best_params)
print("Best score:", study.best_value)

[I 2025-11-27 15:02:47,891] A new study created in memory with name: no-name-1bb98da6-add3-4ae4-aa2f-30b63266008d



############################################################
batch_size: 32 | n_epochs:400 | source: sphere | lr: 9.213596975542375e-05 | weight_decay: 0.0009141326831893455

 step 0 -> loss : 0.00695
 step 80 -> loss : 0.00444
 step 160 -> loss : 0.00433
 step 240 -> loss : 0.00417
 step 320 -> loss : 0.00439


[I 2025-11-27 15:02:51,817] Trial 0 finished with value: 1.52 and parameters: {'batch_size': 32, 'n_epochs': 400, 'source': 'sphere', 'lr': 9.213596975542375e-05, 'weight_decay': 0.0009141326831893455}. Best is trial 0 with value: 1.52.


AUC: 0.9400 | FPR@95: 0.1200 | AP: 0.7000
FM --> AUC: 0.9400 | FPR@95: 0.1200 | AP: 0.7000 | SCORE:  1.5200

############################################################
batch_size: 32 | n_epochs:150 | source: gaussian | lr: 0.00019700645787853734 | weight_decay: 0.0008110698444186423

 step 0 -> loss : 1.04126
 step 30 -> loss : 0.93700
 step 60 -> loss : 0.95838
 step 90 -> loss : 0.87730
 step 120 -> loss : 0.88402


[I 2025-11-27 15:02:53,310] Trial 1 finished with value: 1.7733333333333332 and parameters: {'batch_size': 32, 'n_epochs': 150, 'source': 'gaussian', 'lr': 0.00019700645787853734, 'weight_decay': 0.0008110698444186423}. Best is trial 1 with value: 1.7733333333333332.


...

AUC: 0.9400 | FPR@95: 0.1200 | AP: 0.7000
FM --> AUC: 0.9400 | FPR@95: 0.1200 | AP: 0.7000 | SCORE:  1.5200

############################################################
batch_size: 32 | n_epochs:350 | source: sphere | lr: 1.1299073830641753e-05 | weight_decay: 1.0759340109694175e-05

 step 0 -> loss : 0.00726
 step 70 -> loss : 0.00457
 step 140 -> loss : 0.00443
 step 210 -> loss : 0.00414
 step 280 -> loss : 0.00397


[I 2025-11-27 15:03:30,255] Trial 13 finished with value: 2.0 and parameters: {'batch_size': 32, 'n_epochs': 350, 'source': 'sphere', 'lr': 1.1299073830641753e-05, 'weight_decay': 1.0759340109694175e-05}. Best is trial 2 with value: 2.0.


AUC: 1.0000 | FPR@95: 0.0000 | AP: 1.0000
FM --> AUC: 1.0000 | FPR@95: 0.0000 | AP: 1.0000 | SCORE:  2.0000

############################################################
batch_size: 128 | n_epochs:350 | source: sphere-noised | lr: 0.005326922628395807 | weight_decay: 0.00034274132446837006

 step 0 -> loss : 0.06692
 step 70 -> loss : 0.06675
 step 140 -> loss : 0.06613
 step 210 -> loss : 0.06697
 step 280 -> loss : 0.06722


[I 2025-11-27 15:03:32,303] Trial 14 finished with value: 1.4266666666666665 and parameters: {'batch_size': 128, 'n_epochs': 350, 'source': 'sphere-noised', 'lr': 0.005326922628395807, 'weight_decay': 0.00034274132446837006}. Best is trial 2 with value: 2.0.


AUC: 0.9200 | FPR@95: 0.1600 | AP: 0.6667
FM --> AUC: 0.9200 | FPR@95: 0.1600 | AP: 0.6667 | SCORE:  1.4267
Best hyperparameters: {'batch_size': 32, 'n_epochs': 500, 'source': 'gaussian', 'lr': 0.0018748212842902656, 'weight_decay': 8.587632095977649e-05}
Best score: 2.0


In [283]:
inlier_topic = 'World'
dataset_name = 'agnews'
type_tac = 'fate' 
anomaly_rate = 0.1

train_inlier_agnews, _, _, _ = train_test_val_split(train_agnews_, test_agnews_, inlier_topic, dataset_name, type_tac, anomaly_rate, False)

model_name = 'all-MiniLM-L6-v2'
sentencebertEncoder = embedding_encoder.EmbeddingEncoder(model_name, 'sentencebert', device)

train_inlier_reuters_emb = sentencebertEncoder.forward(train_inlier_reuters)
X_inlier = Tensor(train_inlier_reuters_emb['sbert_embeddings']).to(device)


list_auc_fm = []
list_fpr_fm = []
list_ap_fm = []
list_auc_ocsvm = []
list_fpr_ocsvm = []
list_ap_ocsvm = []

for i in range(5):
    
    print("\n##################################")
    print(f"Loading Dataset for the run {i+1}")
    
    inlier_topic = 'World'
    dataset_name = 'agnews'
    type_tac = 'fate' 
    anomaly_rate = 0.1

    _, train_anomaly_agnews, val_agnews, test_agnews = train_test_val_split(train_agnews_, test_agnews_, inlier_topic, dataset_name, type_tac, anomaly_rate, False)
    
    # print("A sample for valset : ")
    # print(val_20ng[0]['text'])
    # print()
    # print("\nVALSET")
    # print(val_20ng.num_rows)
    # print()
    print("A sample for testset : ")
    print(test_reuters[-1]['text'][:50])
    print()
    # print("TESTSET")
    # print(test_20ng.num_rows)

    

    test_agnews_emb = sentencebertEncoder.forward(test_agnews)
    # val_reuters_emb = sentencebertEncoder.forward(val_reuters)

    # print(X_inlier.shape)

    X_test =  Tensor(test_agnews_emb['sbert_embeddings']).to(device)
    y_test = np.array(test_agnews_emb['anomaly_class'])
    # print(X_test.shape, y_test.shape)
    
    # X_val =  Tensor(val_reuters_emb['sbert_embeddings']).to(device)
    # y_val = np.array(val_reuters_emb['anomaly_class'])
    # print(X_val.shape, y_val.shape)
    
    #########################################
    ################# OCSVM #################
    #########################################  
    
    tac = time.time()
    
    ocsvm_kwargs = {
        "nu": 0.1,
        "kernel": 'rbf',
        "gamma": 'scale'
        }
    clf, _, _ = ocsvm.One_Class_SVM(X_inlier.cpu(), ocsvm_kwargs)
    
    tic = time.time()
    
    print(f" OCSVM finishing... after {(tic-tac)/60:.3f} mn")

    _ = clf.predict(X_test.cpu().detach())           
    scores_test = clf.decision_function(X_test.cpu())

    auc_ocsvm, ap_ocsvm, fpr95_ocsvm = ev.evaluation(y_test, scores_test, verbose=False)
    print(f"OCSVM --> AUC: {auc_ocsvm:.4f} | FPR@95: {fpr95_ocsvm:.4f} | AP: {ap_ocsvm:.4f}\n")
    
    list_auc_ocsvm.append(auc_ocsvm)
    list_fpr_ocsvm.append(fpr95_ocsvm)    
    list_ap_ocsvm.append(ap_ocsvm)  
    
    
    
    #################################################
    ################# FLOW MATCHING #################
    #################################################
    
    batch_size = 64
    # batch_size = study.best_params['batch_size']
    X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

    input_dim = X_inlier.shape[1]
    latent_dim = 256
    sinu = False
    lr = 1e-4
    # lr = study.best_params['lr']
    weight_decay = 1e-5
    # weight_decay = study.best_params['weight_decay']
    n_epochs = 200
    # n_epochs = study.best_params['n_epochs']


    target = X_inlier.cpu()
    source = 'sphere'
    # source = study.best_params['source']


    flow_model = flow_matching.FlowMatching(source, target, input_dim, latent_dim, sinu, device).to(device)
    optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=True)
    
    tac = time.time()

    flow_model_trained = fm_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')
    
    tic = time.time()
    
    print(f" FM finishing... after {(tic-tac)/60:.3f} mn")

    auc, fpr95, ap = fm_trainer.test(X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10)
    
    list_auc_fm.append(auc)
    list_fpr_fm.append(fpr95)    
    list_ap_fm.append(ap)    

AttributeError: 'float' object has no attribute 'textual_anomaly_contamination'

In [248]:
save_results(
    dataset_name=dataset_name,
    inlier_topic=inlier_topic,
    type_emb="sentence_bert",
    ad_model="flow-matching",
    auc_mean=np.mean(list_auc_fm),
    ap_mean=np.mean(list_ap_fm),
    fpr_mean=np.mean(list_fpr_fm),
    auc_std = np.std(list_auc_fm),
    ap_std =  np.std(list_ap_fm),
    fpr_std = np.std(list_fpr_fm) 
)

Nouveaux résultats ajoutés pour (reuters, crude, sentence_bert, flow-matching).


In [249]:
save_results(
    dataset_name=dataset_name,
    inlier_topic=inlier_topic,
    type_emb="sentence_bert",
    ad_model="ocsvm",
    auc_mean=np.mean(list_auc_ocsvm),
    ap_mean=np.mean(list_ap_ocsvm),
    fpr_mean=np.mean(list_fpr_ocsvm),
    auc_std = np.std(list_auc_ocsvm),
    ap_std =  np.std(list_ap_ocsvm),
    fpr_std = np.std(list_fpr_ocsvm) 
)

Nouveaux résultats ajoutés pour (reuters, crude, sentence_bert, ocsvm).


In [279]:
batch_size = 64
# batch_size = study.best_params['batch_size']
X_inlier_dl = DataLoader(TensorDataset(X_inlier), batch_size=batch_size, shuffle=True)

input_dim = X_inlier.shape[1]
latent_dim = 256
sinu = False
lr = 1e-4
# lr = study.best_params['lr']
weight_decay = 1e-5
# weight_decay = study.best_params['weight_decay']
n_epochs = 200
# n_epochs = study.best_params['n_epochs']

target = X_inlier.cpu()
source = 'sphere'
# source = study.best_params['source']

flow_model = flow_matching.FlowMatching(source, target, input_dim, latent_dim, sinu, device).to(device)
optimizer = torch.optim.Adam(flow_model.parameters(), lr=lr, weight_decay=weight_decay)
loss_fn = nn.MSELoss()

fm_trainer = flow_matching.FlowMatchingTrainer(flow_model, verbose=True)

flow_model_trained = fm_trainer.train(X_inlier_dl, lr, weight_decay, loss_fn, n_epochs, optimizer_type='adam')

auc, fpr95, ap = fm_trainer.test(X_test, y_test, score_type='norm', solver_type='midpoint', n_steps=10)

 step 0 -> loss : 0.00477
 step 40 -> loss : 0.00467
 step 80 -> loss : 0.00467
 step 120 -> loss : 0.00460
 step 160 -> loss : 0.00456
AUC: 0.9267 | FPR@95: 0.2343 | AP: 0.5057


## Baselines

### OCSVM

In [278]:
ocsvm_kwargs = {
        "nu": 0.1,
        "kernel": 'rbf',
        "gamma": 'scale'
        }
clf, _, _ = ocsvm.One_Class_SVM(X_inlier.cpu().detach(), ocsvm_kwargs)

_ = clf.predict(X_test.cpu().detach())           
scores_test = clf.decision_function(X_test.cpu().detach())

auc, ap, fpr95 = ev.evaluation(y_test, scores_test, verbose=False)
print(f"AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")

AUC: 0.8440 | FPR@95: 0.6250 | AP: 0.4397


### CVDD

In [14]:
type_emb = 'glove'
emb_model = 'distilbert-base-uncased'
attention_size = 150
n_attention_heads = 10
lr = 0.01
lr_milestones = (5, 8)
n_epochs = 10
lambda_p = 1.0
alpha_scheduler = 'logarithmic'

In [15]:
if type_emb == 'bert':
    tokenizer = AutoTokenizer.from_pretrained(emb_model)
    vocab = None

elif type_emb in ('glove', 'fasttext'):
    corpus = train_inlier_dl_20ng['text']
    vocab = build_vocab(corpus,min_freq=1)
    tokenizer = None

In [16]:
cvdd_model, dl_train, dl_test = cvdd_model_pipeline(train_inlier_dl_20ng, test_dl_20ng, attention_size, n_attention_heads, 
                                               type_emb, 500, 64, True, device, tokenizer, vocab)

In [17]:
cvdd_trainer = cvdd_Net.CVDDTrainer(optimizer_name='adam', learning_rate=lr, lr_milestones=lr_milestones,
                                    n_epochs=n_epochs, lambda_p=lambda_p,
                                    alpha_scheduler=alpha_scheduler, weight_decay=1e-4, device=device)

model_trained = cvdd_trainer.train(cvdd_model, dl_train)

Starting training...
KMean starts
KMeans finish
| Epoch: 001/010 | Train Time: 0.517s | Train Loss: 0.139542 |
| Epoch: 002/010 | Train Time: 0.444s | Train Loss: 0.056700 |
| Epoch: 003/010 | Train Time: 0.441s | Train Loss: 0.049708 |
| Epoch: 004/010 | Train Time: 0.439s | Train Loss: 0.046049 |
| Epoch: 005/010 | Train Time: 0.441s | Train Loss: 0.042732 |
| Epoch: 006/010 | Train Time: 0.439s | Train Loss: 0.041651 |
| Epoch: 007/010 | Train Time: 0.439s | Train Loss: 0.040930 |
| Epoch: 008/010 | Train Time: 0.439s | Train Loss: 0.040041 |
| Epoch: 009/010 | Train Time: 0.439s | Train Loss: 0.039830 |
| Epoch: 010/010 | Train Time: 0.441s | Train Loss: 0.039731 |
Training Time: 5.272s
Finished training. 



In [18]:
auc, ap, fpr95, _ = cvdd_trainer.test(model_trained, dl_test, ad_score='context_dist_mean')
print(f"AUC: {auc:.4f} | FPR@95: {fpr95:.4f} | AP: {ap:.4f}")

AUC: 0.5008 | FPR@95: 0.9654 | AP: 0.1162
